# 10 · Flying a Trajectory

### Recap & why now
Notebook 09 ended on an uncomfortable detail: the motors saturated on every step
command. That is not a tuning failure. A step is a request to be somewhere else
*instantly* — infinite velocity, infinite acceleration — and the vehicle answers the
only way it can, by going flat out and falling short.

Real vehicles are given **trajectories**: a position for every instant, together with
the velocity and acceleration that go with it. This notebook adds that last box, and
closes the project.

### Learning objectives
1. Explain why a step command is physically unreasonable, and what **minimum jerk** fixes.
2. Build a smooth multi-waypoint trajectory with its velocity and acceleration.
3. Feed those derivatives **forward** into the cascade, and measure what they are worth.
4. Test disturbance rejection with a sudden push.
5. State honestly what this simulator does and does not represent.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection that every figure here needs.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print matrices with 3 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=3, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Orientation toolkit, built up over Notebooks 02-05 ==================

def quat_normalize(q):
    """Force |q| = 1. Integration drifts off the unit sphere; this pulls it back."""
    q = np.asarray(q, float)
    return q/np.linalg.norm(q)

def quat_multiply(a, b):
    """Hamilton product a (x) b: 'do b first, then a', the same reading order as matrices."""
    aw, ax, ay, az = a
    bw, bx, by, bz = b
    return np.array([aw*bw - ax*bx - ay*by - az*bz,     # Scalar part.
                     aw*bx + ax*bw + ay*bz - az*by,     # Vector part, x.
                     aw*by - ax*bz + ay*bw + az*bx,     #              y.
                     aw*bz + ax*by - ay*bx + az*bw])    #              z.

def quat_conjugate(q):
    """Flip the vector part — for a unit quaternion this is the INVERSE rotation."""
    return np.array([q[0], -q[1], -q[2], -q[3]])

def quat_to_rotmat(q):
    """The body-to-world rotation matrix that this quaternion represents."""
    w, x, y, z = quat_normalize(q)
    return np.array([[1-2*(y*y+z*z),   2*(x*y-w*z),   2*(x*z+w*y)],
                     [  2*(x*y+w*z), 1-2*(x*x+z*z),   2*(y*z-w*x)],
                     [  2*(x*z-w*y),   2*(y*z+w*x), 1-2*(x*x+y*y)]])

def euler_to_quat(roll, pitch, yaw):
    """ZYX Euler angles -> quaternion. Used to SET a pose, never to store one."""
    cr, sr = np.cos(roll/2), np.sin(roll/2)
    cp, sp = np.cos(pitch/2), np.sin(pitch/2)
    cy, sy = np.cos(yaw/2), np.sin(yaw/2)
    return np.array([cr*cp*cy + sr*sp*sy, sr*cp*cy - cr*sp*sy,
                     cr*sp*cy + sr*cp*sy, cr*cp*sy - sr*sp*cy])

def quat_to_euler(q):
    """Quaternion -> roll, pitch, yaw. For DISPLAY only — never as simulator state."""
    w, x, y, z = quat_normalize(q)
    return np.array([np.arctan2(2*(w*x + y*z), 1 - 2*(x*x + y*y)),
                     np.arcsin(np.clip(2*(w*y - z*x), -1, 1)),      # clip guards against 1+1e-16.
                     np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))])

def quat_from_rotmat(R):
    """Rotation matrix -> quaternion. Four branches, so we never divide by a small number."""
    tr = np.trace(R)
    if tr > 0:
        s_ = np.sqrt(tr + 1.0)*2
        q = np.array([0.25*s_, (R[2,1]-R[1,2])/s_, (R[0,2]-R[2,0])/s_, (R[1,0]-R[0,1])/s_])
    elif R[0,0] > R[1,1] and R[0,0] > R[2,2]:
        s_ = np.sqrt(1.0 + R[0,0] - R[1,1] - R[2,2])*2
        q = np.array([(R[2,1]-R[1,2])/s_, 0.25*s_, (R[0,1]+R[1,0])/s_, (R[0,2]+R[2,0])/s_])
    elif R[1,1] > R[2,2]:
        s_ = np.sqrt(1.0 + R[1,1] - R[0,0] - R[2,2])*2
        q = np.array([(R[0,2]-R[2,0])/s_, (R[0,1]+R[1,0])/s_, 0.25*s_, (R[1,2]+R[2,1])/s_])
    else:
        s_ = np.sqrt(1.0 + R[2,2] - R[0,0] - R[1,1])*2
        q = np.array([(R[1,0]-R[0,1])/s_, (R[0,2]+R[2,0])/s_, (R[1,2]+R[2,1])/s_, 0.25*s_])
    return quat_normalize(q)

def axis_angle_to_quat(axis, angle):
    """Build a quaternion from 'rotate by `angle` about `axis`' — the geometric reading."""
    axis = np.asarray(axis, float); axis = axis/np.linalg.norm(axis)
    return np.array([np.cos(angle/2), *(axis*np.sin(angle/2))])

def quat_rotate(q, v):
    """Rotate v from the body frame into the world frame, using the sandwich product."""
    return quat_multiply(quat_multiply(q, np.array([0.0, *v])), quat_conjugate(q))[1:]

# === The vehicle, and how to draw it =====================================

PARAMS = dict(m=1.0, L=0.25,                       # Mass [kg] and hub-to-rotor distance [m].
              I=np.diag([0.01, 0.01, 0.02]),       # Inertia [kg m^2]; yaw is the heavy axis.
              d=0.016,                             # Drag torque per newton of thrust [m].
              T_min=0.0, T_max=6.0)                # What one motor can produce [N].
g = 9.81                                           # Gravity [m/s^2], along world -z.

ARM = PARAMS["L"]/np.sqrt(2)                       # Each rotor sits ARM along body x AND body y.
MOTOR_POS = np.array([[ ARM, -ARM, 0.0],           # M1 front-right.
                      [ ARM,  ARM, 0.0],           # M2 front-left.
                      [-ARM,  ARM, 0.0],           # M3 rear-left.
                      [-ARM, -ARM, 0.0]])          # M4 rear-right.
SPIN = np.array([-1.0, 1.0, -1.0, 1.0])            # +1 = counter-clockwise seen from above.

def draw_quad(ax, position, q, scale=3.0, thrusts=None):
    """Draw the drone: four arms, four rotors, a nose marker and the thrust arrow."""
    R = quat_to_rotmat(q)                          # Body-to-world, so body points become world points.
    for i, mp in enumerate(MOTOR_POS):
        tip = np.asarray(position, float) + R @ (mp*scale)
        seg = np.array([position, tip])
        ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], color="0.35", lw=2)
        shade = "C3" if i in (0, 1) else "C0"      # Front rotors red, rear blue, so the nose is visible.
        if thrusts is not None:
            load = np.clip(thrusts[i]/PARAMS["T_max"], 0, 1)
            shade = plt.cm.YlOrRd(0.3 + 0.7*load)  # Colour by how hard the motor is working.
        ax.plot([tip[0]], [tip[1]], [tip[2]], "o", ms=6, color=shade)
    ax.quiver(*position, *(R[:, 2]*0.9), color="C1", lw=2.2, arrow_length_ratio=0.25)

def set_3d(ax, xlim, ylim, zlim):
    """Equal-ish 3-D axes with explicit limits, so animations do not jitter."""
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
    ax.set_box_aspect([xlim[1]-xlim[0], ylim[1]-ylim[0], zlim[1]-zlim[0]])
    ax.set_xlabel("x — East [m]"); ax.set_ylabel("y — North [m]"); ax.set_zlabel("z — Up [m]")

print("vehicle ready: %.1f kg, hover %.2f N total, %.3f N per motor, thrust/weight %.2f" %
      (PARAMS["m"], PARAMS["m"]*g, PARAMS["m"]*g/4, 4*PARAMS["T_max"]/(PARAMS["m"]*g)))

# === Mixing and 6-DOF dynamics, from Notebooks 06-07 =====================

POS, VEL, QUAT, OMEGA = slice(0, 3), slice(3, 6), slice(6, 10), slice(10, 13)
MIX = np.vstack([np.ones(4), MOTOR_POS[:, 1], -MOTOR_POS[:, 0], -SPIN*PARAMS["d"]])

def motor_mixer(total_thrust, torques, p=PARAMS):
    """Desired wrench -> four motor thrusts, clipped to what the hardware can do."""
    T4 = np.linalg.solve(MIX, np.array([total_thrust, *torques], float))
    return np.clip(T4, p["T_min"], p["T_max"])     # A propeller cannot pull, nor push forever.

def quad_dynamics(state, motor_thrusts, p=PARAMS, f_ext=np.zeros(3)):
    """x_dot for the 13-state quadcopter, driven by four motor thrusts."""
    q = quat_normalize(state[QUAT]); w = state[OMEGA]
    T, tx, ty, tz = MIX @ np.asarray(motor_thrusts, float)          # Geometry does its job here.
    v_dot = (quat_to_rotmat(q) @ np.array([0.0, 0.0, T])            # Thrust, body -> world.
             + np.array([0.0, 0.0, -p["m"]*g]) + f_ext)/p["m"]      # Gravity, ENU, plus any push.
    q_dot = 0.5*quat_multiply(q, np.array([0.0, *w]))               # Notebook 05's kinematics.
    w_dot = np.linalg.solve(p["I"], np.array([tx, ty, tz]) - np.cross(w, p["I"] @ w))
    return np.concatenate([state[VEL], v_dot, q_dot, w_dot])

def rk4_step(state, motor_thrusts, dt, p=PARAMS, f_ext=np.zeros(3)):
    """One RK4 step, followed by the renormalisation Notebook 05 insisted on."""
    k1 = quad_dynamics(state, motor_thrusts, p, f_ext)
    k2 = quad_dynamics(state + 0.5*dt*k1, motor_thrusts, p, f_ext)
    k3 = quad_dynamics(state + 0.5*dt*k2, motor_thrusts, p, f_ext)
    k4 = quad_dynamics(state + dt*k3, motor_thrusts, p, f_ext)
    s = state + dt/6*(k1 + 2*k2 + 2*k3 + k4)
    s[QUAT] = quat_normalize(s[QUAT])
    return s

def make_state(p=(0, 0, 0), v=(0, 0, 0), q=(1, 0, 0, 0), w=(0, 0, 0)):
    """Assemble the 13-element state vector."""
    return np.concatenate([p, v, q, w]).astype(float)

def simulate(command, T_end=4.0, dt=0.005, s0=None, p=PARAMS, f_ext=lambda t: np.zeros(3)):
    """Fly the drone. `command(t, state)` returns four motor thrusts in newtons."""
    s = make_state() if s0 is None else np.array(s0, float)
    ts, xs, ms = [0.0], [s.copy()], []
    for k in range(int(round(T_end/dt))):
        T4 = np.clip(np.asarray(command(k*dt, s), float), p["T_min"], p["T_max"])
        s = rk4_step(s, T4, dt, p, f_ext(k*dt))
        ts.append((k+1)*dt); xs.append(s.copy()); ms.append(T4)
    return np.array(ts), np.array(xs), np.array(ms)

T_HOVER = PARAMS["m"]*g                            # Total thrust that exactly cancels weight.
HOVER_EACH = T_HOVER/4                             # ...split over four identical motors.
print("model ready — hover needs %.4f N total, %.4f N per motor" % (T_HOVER, HOVER_EACH))

In [ ]:
# === The cascade, built in Notebooks 08-09 ===============================

GAINS = dict(Kp=np.array([2.0, 2.0, 3.0]),         # Position -> velocity; z is stiffer.
             Kv=np.array([4.0, 4.0, 5.0]),         # Velocity -> acceleration.
             K_R=np.array([12.0, 12.0, 6.0]),      # Attitude -> angular rate; yaw softer.
             K_w=np.array([0.06, 0.06, 0.06]),     # Angular rate -> torque.
             v_max=4.0,                            # Speed clamp [m/s].
             tilt_max=np.deg2rad(35))              # How far the COMMAND may ask the drone to lean.

def acc_to_thrust_attitude(a_cmd, yaw_des, q, p=PARAMS, K=GAINS):
    """The pivot: desired acceleration -> (thrust magnitude, desired attitude)."""
    F = p["m"]*(a_cmd + np.array([0.0, 0.0, g]))   # ENU: +g compensates gravity.
    fz = max(F[2], 0.4*p["m"]*g)                   # Never let the vertical part collapse.
    fxy = F[:2]; max_xy = fz*np.tan(K["tilt_max"])
    if np.linalg.norm(fxy) > max_xy:
        fxy = fxy*max_xy/np.linalg.norm(fxy)       # Cap the tilt the command may request.
    F = np.array([fxy[0], fxy[1], fz])
    T = float(F @ quat_to_rotmat(q)[:, 2])         # Project onto the axis we can actually push along.
    z_des = F/np.linalg.norm(F)
    x_c = np.array([np.cos(yaw_des), np.sin(yaw_des), 0.0])
    y_des = np.cross(z_des, x_c); y_des /= np.linalg.norm(y_des)
    return T, quat_from_rotmat(np.column_stack([np.cross(y_des, z_des), y_des, z_des]))

def attitude_controller(q, q_des, K=GAINS):
    """Orientation error -> desired body angular rate."""
    q_e = quat_multiply(quat_conjugate(q), q_des)  # Rotation from current TO desired, in body axes.
    if q_e[0] < 0:
        q_e = -q_e                                 # Short way round — Notebook 04, Section 4.
    return 2.0*K["K_R"]*q_e[1:]

def rate_controller(w_cmd, w, p=PARAMS, K=GAINS):
    """Angular-rate error -> body torque, with the gyroscopic term fed forward."""
    return K["K_w"]*(w_cmd - w) + np.cross(w, p["I"] @ w)

def cascaded_controller(state, p_des, v_ff, a_ff, yaw_des, p=PARAMS, K=GAINS):
    """One pass through the whole stack: state + reference -> four motor thrusts."""
    v_cmd = K["Kp"]*(p_des - state[POS]) + v_ff
    speed = np.linalg.norm(v_cmd)
    if speed > K["v_max"]:
        v_cmd = v_cmd*K["v_max"]/speed             # Keep the direction, cap the magnitude.
    a_cmd = K["Kv"]*(v_cmd - state[VEL]) + a_ff
    T, q_des = acc_to_thrust_attitude(a_cmd, yaw_des, state[QUAT], p, K)
    w_cmd = attitude_controller(state[QUAT], q_des, K)
    tau = rate_controller(w_cmd, state[OMEGA], p, K)
    return motor_mixer(T, tau, p)

def fly(reference, T_end, dt=0.005, s0=None, p=PARAMS, K=GAINS,
        yaw_of_t=lambda t: 0.0, f_ext=lambda t: np.zeros(3)):
    """Closed-loop flight. `reference(t)` returns (p_des, v_ff, a_ff)."""
    s = make_state() if s0 is None else np.array(s0, float)
    ts, xs, ms, rs = [0.0], [s.copy()], [], []
    for k in range(int(round(T_end/dt))):
        p_des, v_ff, a_ff = reference(k*dt)
        T4 = cascaded_controller(s, p_des, v_ff, a_ff, yaw_of_t(k*dt), p, K)
        s = rk4_step(s, T4, dt, p, f_ext(k*dt))
        ts.append((k+1)*dt); xs.append(s.copy()); ms.append(T4); rs.append(p_des)
    return np.array(ts), np.array(xs), np.array(ms), np.array(rs)

hold = lambda target: (lambda t: (np.array(target, float), np.zeros(3), np.zeros(3)))
print("cascade loaded — six small controllers, each feeding the next")

## 1 · A reference the drone can actually follow

The standard building block is the **minimum-jerk** polynomial, which moves from 0 to 1
over a normalised time $\tau = t/T$:

$$s(\tau) = 10\tau^3 - 15\tau^4 + 6\tau^5$$

The five coefficients are exactly what is needed to satisfy six boundary conditions —
position, velocity **and** acceleration all zero at both ends. That last pair is the one
that matters for a quadcopter: acceleration means tilt, and starting a segment with
non-zero acceleration would demand the drone already be tilted at that instant.

In [ ]:
def min_jerk(tau):
    """The 0->1 profile and its first two derivatives, in normalised time."""
    s = 10*tau**3 - 15*tau**4 + 6*tau**5           # Position.
    sd = 30*tau**2 - 60*tau**3 + 30*tau**4         # d s / d tau.
    sdd = 60*tau - 180*tau**2 + 120*tau**3         # d^2 s / d tau^2.
    return s, sd, sdd

def waypoint_trajectory(waypoints, durations):
    """Chain minimum-jerk segments; returns ref(t) -> (p_des, v_des, a_des) and the total time."""
    wp = [np.asarray(w, float) for w in waypoints]
    edges = np.concatenate([[0.0], np.cumsum(durations)])
    def ref(t):
        if t <= 0:         return wp[0].copy(), np.zeros(3), np.zeros(3)
        if t >= edges[-1]: return wp[-1].copy(), np.zeros(3), np.zeros(3)   # Hold the last point.
        i = int(np.searchsorted(edges, t, side="right") - 1)
        T = durations[i]; tau = (t - edges[i])/T
        s, sd, sdd = min_jerk(tau)
        d = wp[i+1] - wp[i]
        return wp[i] + d*s, d*sd/T, d*sdd/T**2      # Chain rule: each derivative brings a 1/T.
    return ref, float(edges[-1])

tau = np.linspace(0, 1, 300)
s_, sd_, sdd_ = min_jerk(tau)
fig, axes = plt.subplots(1, 3, figsize=(13, 2.8))
for ax_, y, name in zip(axes, [s_, sd_, sdd_], ["position", "velocity", "acceleration"]):
    ax_.plot(tau, y, color="C0", lw=2.2); ax_.set_xlabel(r"$\tau$"); ax_.set_title(name, fontsize=10)
axes[0].plot([0, 1], [0, 1], color="0.7", ls="--", lw=1.4)
plt.tight_layout(); plt.show()
print("boundary check: s(0) = %.0f, s(1) = %.0f, s'(0) = %.0f, s'(1) = %.0f, s''(0) = %.0f, s''(1) = %.0f ✔" %
      (s_[0], s_[-1], sd_[0], sd_[-1], sdd_[0], sdd_[-1]))

## 2 · The mission

Climb, fly East, fly North, climb again, and return home — five segments, each with its
own duration:

```text
(0,0,0) → (0,0,1.5) → (2,0,1.5) → (2,2,1.5) → (2,2,2.5) → (0,0,1.5)
   2.5 s      3.0 s      3.0 s      2.0 s      4.0 s
```

The generator returns **three** things at every instant. The last two are what make this
more than a moving setpoint: they tell the controller what the vehicle *should already
be doing*, so feedback only has to correct the difference.

In [ ]:
WAYPOINTS = [(0, 0, 0), (0, 0, 1.5), (2.0, 0, 1.5), (2.0, 2.0, 1.5), (2.0, 2.0, 2.5), (0, 0, 1.5)]
DURATIONS = [2.5, 3.0, 3.0, 2.0, 4.0]
ref_fn, T_traj = waypoint_trajectory(WAYPOINTS, DURATIONS)

grid = np.linspace(0, T_traj, 500)
P = np.array([ref_fn(t_)[0] for t_ in grid])
V = np.array([ref_fn(t_)[1] for t_ in grid])
A = np.array([ref_fn(t_)[2] for t_ in grid])
print("duration %.1f s, peak commanded speed %.2f m/s, peak acceleration %.2f m/s^2" %
      (T_traj, np.linalg.norm(V, axis=1).max(), np.linalg.norm(A, axis=1).max()))
print("implied peak tilt if that acceleration were horizontal: %.1f° — well inside the %.0f° limit" %
      (np.degrees(np.arctan(np.linalg.norm(A[:, :2], axis=1).max()/g)), np.degrees(GAINS["tilt_max"])))

fig = plt.figure(figsize=(12.5, 3.6))
ax = fig.add_subplot(121, projection="3d")
ax.plot(P[:, 0], P[:, 1], P[:, 2], color="C2", lw=2.4)
W = np.array(WAYPOINTS, float)
ax.plot(W[:, 0], W[:, 1], W[:, 2], "*", color="C3", ms=11)
set_3d(ax, (-0.5, 2.8), (-0.5, 2.8), (0, 3)); ax.set_title("The reference path", fontsize=10)
ax.view_init(elev=24, azim=-62)
ax2 = fig.add_subplot(122)
for i, lbl in enumerate(["$v_x$", "$v_y$", "$v_z$"]):
    ax2.plot(grid, V[:, i], lw=1.7, label=lbl)
ax2.set_xlabel("time [s]"); ax2.set_ylabel("velocity [m/s]"); ax2.legend(fontsize=8)
ax2.set_title("The feedforward the controller receives")
plt.tight_layout(); plt.show()

## 3 · Flying it, and what feedforward is worth

The controller is untouched — the same six functions from Notebook 09. The only
difference is that $v_{ff}$ and $a_{ff}$ now carry the trajectory's derivatives instead
of zeros.

The cell below flies the identical path twice: once with the derivatives, once with them
zeroed so the controller sees only a moving setpoint.

In [ ]:
t, X, M, R = fly(ref_fn, T_end=T_traj + 3.0)
err = np.linalg.norm(X[1:, POS] - R, axis=1)
rpy = np.degrees(np.array([quat_to_euler(q_) for q_ in X[:, QUAT]]))

ref_no_ff = lambda t_: (ref_fn(t_)[0], np.zeros(3), np.zeros(3))    # Same path, no derivatives.
t2, X2, M2, R2 = fly(ref_no_ff, T_end=T_traj + 3.0)
err2 = np.linalg.norm(X2[1:, POS] - R2, axis=1)

print("                      RMS error     worst error   peak tilt   motors")
print("  with feedforward %11.4f m %13.4f m %9.1f° %6.2f-%.2f N" %
      (np.sqrt(np.mean(err**2)), err.max(), np.abs(rpy[:, :2]).max(), M.min(), M.max()))
print("  position only    %11.4f m %13.4f m" % (np.sqrt(np.mean(err2**2)), err2.max()))
print("\n  feedforward improves RMS tracking by a factor of %.0f." %
      (np.sqrt(np.mean(err2**2))/np.sqrt(np.mean(err**2))))
print("  motors saturated %.1f%% of the flight — compare Notebook 09's step commands." %
      (100*np.mean(M.max(axis=1) >= PARAMS["T_max"]-1e-9)))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 3.4))
a1.plot(t[1:], err, color="C0", lw=1.6, label="with feedforward")
a1.plot(t2[1:], err2, color="C3", lw=1.6, label="position only")
a1.set_xlabel("time [s]"); a1.set_ylabel("tracking error [m]"); a1.legend(fontsize=8)
a1.set_title("The cost of not saying what is coming")
a2.plot(R[:, 0], R[:, 1], color="C2", ls="--", lw=2, label="desired")
a2.plot(X[:, 0], X[:, 1], color="C0", lw=1.6, label="with feedforward")
a2.plot(X2[:, 0], X2[:, 1], color="C3", lw=1.4, label="position only")
a2.set_xlabel("x [m]"); a2.set_ylabel("y [m]"); a2.set_aspect("equal"); a2.legend(fontsize=8)
a2.set_title("Top view: corner-cutting without it")
plt.tight_layout(); plt.show()

## 4 · A shove in mid-flight

Feedback earns its keep when something happens the trajectory did not predict. A 3 N
push — about 30% of the vehicle's weight — for half a second, in the middle of the
mission.

All that unused authority from Section 3 is what pays for the recovery.

In [ ]:
gust = lambda t_: np.array([3.0, 0.0, 0.0]) if 5.0 <= t_ < 5.5 else np.zeros(3)
t_g, X_g, M_g, R_g = fly(ref_fn, T_end=T_traj + 3.0, f_ext=gust)
e_g = np.linalg.norm(X_g[1:, POS] - R_g, axis=1)
after = t_g[1:] >= 5.0
peak_t = t_g[1:][after][np.argmax(e_g[after])]
back = (e_g[after] < 0.05) & (t_g[1:][after] > peak_t)

print("undisturbed RMS error %.4f m, disturbed %.4f m" %
      (np.sqrt(np.mean(err**2)), np.sqrt(np.mean(e_g**2))))
print("peak error after the push %.3f m at t = %.2f s (the push ended at 5.5 s)" %
      (e_g[after].max(), peak_t))
print("back inside 5 cm at t = %.2f s, %.2f s after the peak" %
      (t_g[1:][after][np.argmax(back)], t_g[1:][after][np.argmax(back)] - peak_t))
print("peak tilt during the recovery %.1f°, peak motor %.2f N of %.1f available" %
      (np.abs(np.degrees(np.array([quat_to_euler(q_) for q_ in X_g[:, QUAT]]))[:, :2]).max(),
       M_g.max(), PARAMS["T_max"]))

fig, ax = plt.subplots(figsize=(7.4, 3.0))
ax.plot(t[1:], err, color="0.6", lw=1.4, label="no disturbance")
ax.plot(t_g[1:], e_g, color="C3", lw=1.7, label="3 N push for 0.5 s")
ax.axvspan(5.0, 5.5, color="C1", alpha=0.25)
ax.set_xlabel("time [s]"); ax.set_ylabel("tracking error [m]"); ax.legend(fontsize=9)
ax.set_title("Blown off the path, then back on it")
plt.show()

## 🧪 Try it yourself

**E1.** The smooth trajectory used a third of the tilt limit and never saturated a
motor, while Notebook 09's step command did both. Both reach the same place. What did
the smooth version actually buy?

**E2.** Halve every segment duration and see what breaks first — tilt, motor saturation,
or tracking error.

In [ ]:
# --- Solution E1 ---
print("E1: headroom. Both trajectories arrive, so on the 'did it get there' test they tie. The")
print("    difference is what is left over: the smooth flight kept motors between %.2f and %.2f N" %
      (M.min(), M.max()))
print("    of the %.1f N available, and peak tilt at %.1f° of the %.0f° limit." %
      (PARAMS["T_max"], np.abs(rpy[:, :2]).max(), np.degrees(GAINS["tilt_max"])))
print("    Section 4 spent exactly that margin absorbing a gust. A drone flown at its limits has")
print("    nothing left for the disturbance it did not plan for — and disturbances are the only")
print("    reason feedback exists.")

# --- Solution E2 ---
print("\nE2:  time scale   duration   peak tilt   saturated   RMS error")
for scale in (1.0, 0.7, 0.5, 0.4, 0.3):
    ref_s, T_s = waypoint_trajectory(WAYPOINTS, [d*scale for d in DURATIONS])
    ts, Xs, Ms, Rs = fly(ref_s, T_end=T_s + 3.0)
    es = np.linalg.norm(Xs[1:, POS] - Rs, axis=1)
    rps = np.degrees(np.array([quat_to_euler(q_) for q_ in Xs[:, QUAT]]))
    print("    %8.1fx %10.1f s %10.1f° %10.1f%% %12.4f m" %
          (scale, T_s, np.abs(rps[:, :2]).max(),
           100*np.mean(Ms.max(axis=1) >= PARAMS["T_max"]-1e-9), np.sqrt(np.mean(es**2))))

print("    Peak acceleration scales as 1/T^2, so halving the durations quadruples the tilt demand.")
print("    Tilt is what runs out first — it reaches the limit well before the motors do, because")
print("    our thrust-to-weight of %.1f is generous. On a heavier vehicle the order would reverse." %
      (4*PARAMS["T_max"]/(PARAMS["m"]*g)))
print("    Either way, the fix is not a better controller: it is a slower trajectory.")

## 🚁 Mini-project: the complete flight

Everything this project built, in one clip: quaternion orientation, 6-DOF dynamics,
motor mixing, the cascade, and a smooth reference. The rotors are shaded by load, so you
can watch the mixer working.

In [ ]:
step = 16
fig = plt.figure(figsize=(7.0, 5.6))
ax = fig.add_subplot(111, projection="3d")

def frame(j):
    ax.clear()
    k = step*j
    ax.plot(R[:, 0], R[:, 1], R[:, 2], color="C2", ls="--", lw=1.5)      # The desired path.
    ax.plot(X[:k+1, 0], X[:k+1, 1], X[:k+1, 2], color="C0", lw=1.8)      # What was flown.
    draw_quad(ax, X[k, POS], X[k, QUAT], scale=2.2, thrusts=M[min(k, len(M)-1)])
    set_3d(ax, (-0.8, 2.8), (-0.8, 2.8), (0, 3.2))
    ax.set_title("t = %5.2f s   error %5.3f m   tilt %4.1f°" %
                 (k*0.005, np.linalg.norm(X[k, POS] - R[min(k, len(R)-1)]),
                  np.abs(rpy[k, :2]).max()), fontsize=10)
    ax.view_init(elev=24, azim=-62)
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(X)//step, interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

## 🤖 What this project built

```text
   01-02   frames, conventions, and rotation in 3-D
              │
   03-05   Euler angles and their hole; quaternions; integrating orientation
              │
   06      6-DOF rigid-body dynamics — thirteen states, two equations
              │
   07      motors and mixing, and what saturation really delivers
              │
   08      the inner loops: attitude and rate, the part that must never stop
              │
   09      the full cascade — position error becomes four motor thrusts
              │
   10      a smooth trajectory, and feedforward worth a factor of twenty
```

> **🤖 Robotics connection.** The number worth remembering is the comparison in Section
> 3. Same vehicle, same controller, same route: a step command saturated the motors and
> demanded 40° of tilt, while the smooth trajectory tracked to about a centimetre using a
> third of the available tilt and never touching a limit. **Most of what looks like
> control performance is actually a question you asked well.**

**This is an educational simulator.** It assumes a rigid symmetric body, instant motors
with no spin-up lag, no aerodynamic drag beyond a torque constant, perfect noiseless
state feedback, and no ground — the drone in Notebook 06 fell to −16 m without
complaint. Project 4 is about how far a real state estimate is from the truth, and
Project 6 replaces this notebook's trajectory generator with an optimised one.